<a href="https://colab.research.google.com/github/AbdulUMSL/EENG-1108/blob/main/Midterm22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# app.py code
import streamlit as st
from vehicles import GroundVehicle, Drone, UGV
import random

# --- Page Config ---
st.set_page_config(page_title="Smart Fleet Control", layout="wide")
st.title("🚜 Smart Fleet Management System")

# --- Initialize Session State ---
if 'fleet' not in st.session_state:
    st.session_state.fleet = {}

# --- Sidebar: Add New Vehicle ---
with st.sidebar:
    st.header("Add to Fleet")
    v_type = st.selectbox("Vehicle Type", ["Ground Vehicle", "Drone", "UGV"])
    v_id = st.text_input("Vehicle ID (Unique)")
    v_model = st.text_input("Model Name")

    if st.button("Deploy Vehicle"):
        if v_id and v_model:
            if v_id not in st.session_state.fleet:
                if v_type == "Ground Vehicle":
                    new_v = GroundVehicle(v_id, v_model)
                elif v_type == "Drone":
                    new_v = Drone(v_id, v_model)
                else:
                    new_v = UGV(v_id, v_model)

                st.session_state.fleet[v_id] = new_v
                st.success(f"{v_type} {v_id} deployed!")
            else:
                st.error("ID already exists!")
        else:
            st.warning("Please fill all fields.")

# --- Main Dashboard ---
col1, col2 = st.columns([2, 1])

with col1:
    st.header("Live Fleet Status")
    if not st.session_state.fleet:
        st.info("No vehicles currently in the fleet.")
    else:
        # Create a table-like view
        for vid, v in st.session_state.fleet.items():
            with st.expander(f"{v.get_battery_icon()} | {vid} - {v.model}"):
                c1, c2, c3 = st.columns(3)
                c1.metric("Battery", f"{int(v.battery)}%")
                c2.write(f"**Type:** {type(v).__name__}")
                c3.write(f"**Status:** {v.status}")

with col2:
    st.header("Control Panel")
    if st.session_state.fleet:
        target_id = st.selectbox("Select Vehicle", list(st.session_state.fleet.keys()))
        target_v = st.session_state.fleet[target_id]

        # Action Buttons
        if st.button("Issue Move Command"):
            dist = random.randint(1, 5)
            target_v.move(dist)
            st.rerun()

        if st.button("Full Charge"):
            target_v.charge()
            st.rerun()

        # Polymorphism in action: Special command for UGV
        if isinstance(target_v, UGV):
            weight = st.slider("Payload Weight (kg)", 1, 20, 5)
            if st.button("Execute Delivery"):
                target_v.deliver(weight)
                st.rerun()
    else:
        st.write("Deploy a vehicle to enable controls.")


In [ ]:
# vehicles.py
import random

class Vehicle:
    def __init__(self, vehicle_id, model):
        self.vehicle_id = vehicle_id
        self.model = model
        self.battery = 100
        self.status = "Idle"

    def charge(self):
        self.battery = 100
        self.status = "Charging Complete"

    def move(self, distance):
        # Base move logic (to be overridden)
        pass

    def get_battery_icon(self):
        if self.battery > 70: return "🔋 Full"
        if self.battery > 20: return "🪫 Medium"
        return "⚠️ Low"

class GroundVehicle(Vehicle):
    def __init__(self, vehicle_id, model, terrain="Road"):
        super().__init__(vehicle_id, model)
        self.terrain = terrain

    def move(self, distance):
        consumption = distance * 8 if self.terrain == "Road" else distance * 12
        if self.battery >= consumption:
            self.battery -= consumption
            self.status = f"Drove {distance}km on {self.terrain}"
        else:
            self.status = "Insufficient Battery"

class Drone(Vehicle):
    def move(self, distance):
        consumption = distance * 15 # High energy demand
        if self.battery < 15:
            self.status = "Critical Battery: Cannot Takeoff"
        elif self.battery >= consumption:
            self.battery -= consumption
            self.status = f"Flew {distance}km"
        else:
            self.status = "Insufficient Battery for flight"

class UGV(Vehicle):
    def move(self, distance):
        consumption = distance * 5
        if self.battery >= consumption:
            self.battery -= consumption
            self.status = f"Patrolled {distance}km"
        else:
            self.status = "Insufficient Battery"

    def deliver(self, weight):
        consumption = weight * 2
        if self.battery >= consumption:
            self.battery -= consumption
            self.status = f"Delivered {weight}kg payload"
        else:
            self.status = "Cargo too heavy for current battery"